In [1]:
# General nectar energetics reference table
# (Pattrick et al. 2025-style conversion: concentration + composition -> J/µL)

import pandas as pd
import numpy as np

In [2]:
# Energetic content (J/mg) for individual sugars (Pattrick et al. 2025, Table 4)
ENERGY_J_PER_MG = {
    "sucrose": 16.48,
    "glucose": 15.56,
    "fructose": 15.60
}

# Density polynomial parameters (Pattrick et al. 2025, Table 2; used in your prior notebook)
A1, A2, A3 = 0.998709, 0.0037430, 0.000017639

def density_g_per_mL(Cw_total_percent):
    """
    Approximate solution density (g/mL) as a function of total sugar concentration (% w/w).
    """
    Cw = float(Cw_total_percent)
    return A1 + A2 * Cw + A3 * (Cw ** 2)

# Per-event energies from your manuscript Table 1 footnote
E_BUZZ_J = 0.0992
E_TAKEOFF_J = 0.0968

# Optional: buzzing-time bout equivalents (using mean buzz duration ~1.59 s)
# Energy per second of buzzing implied by E_BUZZ_J / 1.59
MEAN_BUZZ_DURATION_S = 1.59
E_BUZZ_PER_S_J = E_BUZZ_J / MEAN_BUZZ_DURATION_S  # ~0.0624 J/s

BOUT_SECONDS = [10, 30, 60]

In [3]:
REQUIRED_KEYS = {"sucrose", "glucose", "fructose"}

def normalize_composition(comp):
    """
    Accept either fractions summing to 1 or percentages summing to 100.
    Returns fractions summing to 1.
    """
    if set(comp.keys()) != REQUIRED_KEYS:
        raise ValueError(f"Composition must have exactly these keys: {REQUIRED_KEYS}")

    values = np.array([comp["sucrose"], comp["glucose"], comp["fructose"]], dtype=float)
    s = values.sum()
    if s <= 0:
        raise ValueError("Composition sum must be > 0.")

    # If it looks like percentages (sum ~100), normalize to 1
    if 90 <= s <= 110:
        values = values / 100.0
        s = values.sum()

    # Now enforce ~1
    if abs(s - 1.0) > 1e-6:
        values = values / s

    return {"sucrose": values[0], "glucose": values[1], "fructose": values[2]}

def energy_density_J_per_uL(Cw_total_percent, composition):
    """
    Compute nectar energy density (J/µL) from:
      - Cw_total_percent: total sugar concentration (% w/w)
      - composition: dict (fractions or %) for sucrose/glucose/fructose

    Method consistent with your existing notebook:
      1) density(Cw_total) from polynomial
      2) convert each sugar from % w/w to % w/v via Cv = Cw_sugar * density
      3) energy per µL = sum(Cv_sugar * J/mg_sugar) / 100
         (because Cv is g/100 mL; this collapses unit conversions exactly as in your notebook)
    """
    comp = normalize_composition(composition)
    rho = density_g_per_mL(Cw_total_percent)

    # Per-sugar % w/w contributions
    Cw_suc = Cw_total_percent * comp["sucrose"]
    Cw_glu = Cw_total_percent * comp["glucose"]
    Cw_fru = Cw_total_percent * comp["fructose"]

    # Convert to % w/v (g/100 mL)
    Cv_suc = Cw_suc * rho
    Cv_glu = Cw_glu * rho
    Cv_fru = Cw_fru * rho

    J_per_uL = (
        Cv_suc * ENERGY_J_PER_MG["sucrose"] +
        Cv_glu * ENERGY_J_PER_MG["glucose"] +
        Cv_fru * ENERGY_J_PER_MG["fructose"]
    ) / 100.0

    return float(J_per_uL)

In [4]:
# Concentrations to include in the reference table (% w/w)
CONCENTRATIONS = [20, 40, 60]  # change/add as needed

# Composition scenarios (fractions or % both accepted)
SCENARIOS = {
    "100% sucrose": {"sucrose": 1.0, "glucose": 0.0, "fructose": 0.0},
    "50:50 hexose (G:F)": {"sucrose": 0.0, "glucose": 0.5, "fructose": 0.5},
    "50% sucrose + 25% G + 25% F": {"sucrose": 0.5, "glucose": 0.25, "fructose": 0.25},
}

In [5]:
rows = []
for Cw in CONCENTRATIONS:
    for scen_name, comp in SCENARIOS.items():
        J_uL = energy_density_J_per_uL(Cw, comp)

        row = {
            "Sugar conc. (% w/w)": Cw,
            "Composition scenario": scen_name,
            "Energy density (J/µL)": J_uL,
            "µL per buzz (0.0992 J)": E_BUZZ_J / J_uL,
            "µL per take-off (0.0968 J)": E_TAKEOFF_J / J_uL,
        }

        # Buzzing-time equivalents
        for t in BOUT_SECONDS:
            E_bout = E_BUZZ_PER_S_J * t
            row[f"µL for {t} s buzzing (~{E_bout:.3f} J)"] = E_bout / J_uL

        rows.append(row)

ref_df = pd.DataFrame(rows)

# Round for readability
ref_df["Energy density (J/µL)"] = ref_df["Energy density (J/µL)"].round(3)
for c in ref_df.columns:
    if c.startswith("µL"):
        ref_df[c] = ref_df[c].round(3)

ref_df = ref_df.sort_values(["Sugar conc. (% w/w)", "Composition scenario"])
ref_df

,Sugar conc. (% w/w),Composition scenario,Energy density (J/µL),µL per buzz (0.0992 J),µL per take-off (0.0968 J),µL for 10 s buzzing (~0.624 J),µL for 30 s buzzing (~1.872 J),µL for 60 s buzzing (~3.743 J)
0,20,100% sucrose,3.562,0.028,0.027,0.175,0.526,1.051
2,20,50% sucrose + 25% G + 25% F,3.464,0.029,0.028,0.180,0.540,1.081
1,20,50:50 hexose (G:F),3.367,0.029,0.029,0.185,0.556,1.112
3,40,100% sucrose,7.756,0.013,0.012,0.080,0.241,0.483
5,40,50% sucrose + 25% G + 25% F,7.545,0.013,0.013,0.083,0.248,0.496
4,40,50:50 hexose (G:F),7.333,0.014,0.013,0.085,0.255,0.510
6,60,100% sucrose,12.724,0.008,0.008,0.049,0.147,0.294
8,60,50% sucrose + 25% G + 25% F,12.376,0.008,0.008,0.050,0.151,0.302
7,60,50:50 hexose (G:F),12.029,0.008,0.008,0.052,0.156,0.311


In [7]:
ref_df.to_csv("C:/Users/labadmin/Documents/Uppsala analyses/Manuscript/Submission/Proc B/Reviews/general_nectar_reference_table.csv", index=False)
print("Saved: general_nectar_reference_table.csv")

Saved: general_nectar_reference_table.csv
